In [2]:
"""
- This module contain the code for the full attack to work.
- You don't need to install or load anything,
- You need only a working sagemath server.
- Online testing : https://sagecell.sagemath.org/
"""


def coppersmith_howgrave_univariate(pol, modulus, beta, mm, tt, XX):
    """
    Coppersmith revisited by Seck et al.

    finds a solution if:
    * b|modulus, b >= modulus^beta , 0 < beta <= 1
    * |x| < XX

    # Copyright : Michel Seck
    # Github : https://github.com/mseckept/generalized-wiener-attack

    """
    #
    # init
    #
    dd = pol.degree()
    nn = dd * mm + tt

    #
    # checks
    #
    if not 0 < beta <= 1:
        raise ValueError("beta should belongs in (0, 1]")

    if not pol.is_monic():
        raise ArithmeticError("Polynomial must be monic.")

    # Coppersmith revisited algo for univariate
    # change ring of pol and x
    polZ = pol.change_ring(ZZ)
    x = polZ.parent().gen()
    # compute polynomials
    gg = []
    for ii in range(mm):
        for jj in range(dd):
            gg.append((x * XX)**jj * modulus**(mm - ii) * polZ(x * XX)**ii)
    for ii in range(tt):
        gg.append((x * XX)**ii * polZ(x * XX)**mm)

    # construct lattice B
    BB = Matrix(ZZ, nn)
    for ii in range(nn):
        for jj in range(ii+1):
            BB[ii, jj] = gg[ii][jj]
    # LLL
    BB = BB.LLL()
    # transform shortest vector in polynomial
    new_pol = 0
    for ii in range(nn):
        new_pol += x**ii * BB[0, ii] / XX**ii
    # factor polynomial
    potential_roots = new_pol.roots()
    # test roots
    roots = []
    for root in potential_roots:
        if root[0].is_integer():
            result = polZ(ZZ(root[0]))
            if gcd(modulus, result) >= modulus^beta:
                roots.append(ZZ(root[0]))
    return roots


def get_p(N, pa):
    """Returns a factor p of N or None given N = pq and an approximation of p."""
    F.<x> = PolynomialRing(Zmod(N), implementation='NTL');
    pol = x - pa
    dd = pol.degree()
    beta = 0.5                             # we should have q >= N^beta
    epsilon = beta / 7                     # <= beta/7
    mm = ceil(beta**2 / (dd * epsilon))    # optimized
    tt = floor(dd * mm * ((1/beta) - 1))   # optimized
    XX = ceil(N**((beta**2/dd) - epsilon)) # we should have |diff| < X

    # Coppersmith

    roots = coppersmith_howgrave_univariate(pol, N, beta, mm, tt, XX)
    if roots:
        return pa - roots[0]


def rand_primes(size, mu):
    """Generates random primes with q < p < mu*q."""

    p = random_prime(1 << (size - 1), 1 << size)
    while True:
        q = random_prime(1 << (size - 1), 1 << size)

        if p < q:
            p, q = q, p
            if q < p < mu*q:
                 break
    return p, q


def get_approx_p_pm_q(Nsp1_square, N, e, x, y, prec=1000):
    """Returns an approximate p+q and p-q given N, e, x, y such that

        ex - (p^4 - 1)(q^4 - 1)y = w and N=pq
     """
    # Increase the precision
    RF = RealField(prec)
    # Convert inputs to high-precision real numbers
    Nsp1_square = RF(Nsp1_square)
    N, e, x, y = RF(N), RF(e), RF(x), RF(y)
    # Perform the computation with high precision
    inner_sqrt = sqrt(abs(Nsp1_square - e*x/y))
    p_plus_q = floor(sqrt(abs(2*N + inner_sqrt)))
    p_minus_q = floor(sqrt(abs(-2*N + inner_sqrt)))
    return p_plus_q, p_minus_q


def gen_weak_RSA_instance(nbits, mu, prec=1024):
    """Generates weak public key instances of the RSA-like cryptosystem from
       Seck et al. (AfricaCrypt 2025)
    """
    RF = RealField(prec)
    p, q = rand_primes(nbits//2, mu); N = p*q; phi = (p^4-1)*(q^4-1)
    N_squared = RF(N**2); N_ss = RF(N**4)
    threshold = (2*mu**2*N_ss-(mu**2+1)**2*N_squared + 2*mu**2)/(4*mu**2*(mu**2+1)*N+ 2 * (3*mu**4+4*mu**2+1)*N_squared)
    found = False
    while not found:
        d = randint(1, round(sqrt(threshold)) - 1)
        if gcd(d, phi) == 1:
            found = True
    e = inverse_mod(phi-d, phi)
    return N, e


def factor_N(N, e, mu, prec=None, bug=False):
    """Returns the factors p, q of N or None given the public key (N, e)."""

    i = x = y = p_plus_q = p_minus_q = pa = p = q = None

    try:
        precision = prec if prec else 5 * N.bit_length()

        Nsp1_square = (N**2 + 1)**2

        r = (2 * mu**2 * e) / (
            2 * mu**2 * N**4
            - (mu**2 + 1)**2 * N**2
            + 2 * mu**2
        )

        cf = r.continued_fraction()
        print(f"Continued fraction expansion: {cf}")

        for i, xy in enumerate(cf.convergents()[1:]):
            x = xy.denominator()
            y = xy.numerator()

            p_plus_q, p_minus_q = get_approx_p_pm_q(
                Nsp1_square, N, e, x, y, precision
            )

            pa = (p_plus_q + p_minus_q) // 2

            p = get_p(N, pa)

            if p:
                q = N // p
                return (i, x, y, p_plus_q, p_minus_q, pa, p, q)

    except Exception:
        if bug:
            raise

    return (i, x, y, p_plus_q, p_minus_q, pa, p, q)

# Test
N = 26775501908191905946876668256607628252706474386504699047017
e = 453050078713827481952641242194154220277177895571370108486994492128005609727464231870541236674346742999351986948106354604795069180208060129777145263078104792075428319356342882034554307504611258078457626617783956800491143320938846608793
mu = 15
res = factor_N(N, e, mu, bug=False)
i, x, y, p_plus_q, p_minus_q, pa, p, q = res
print(f"Convergent {i}:\n x = {x}\n y = {y}")
print(f"Approximations: \n p_plus_q = {p_plus_q} \n p_minus_q = {p_minus_q}")
print(f"pa = {pa}")
print(f"Get \np={p} \n and \nq={q}")


Continued fraction expansion: [0; 1, 7, 2, 3, 2, 1, 9, 60, 1, 29, 1, 1, 1, 1, 1, 2, 2, 5, 3, 6, 1, 2, 4, 1, 1, 1, 1, 1, 2, 5, 1, 2, 2, 1, 2, 14, 1, 1, 1, 2, 1, 1, 4, 1, 5, 6, 2, 1, 1, 1, 1, 1, 5, 5, 1, 1, 1, 1, 1, 6, 4, 1, 5, 1, 2, 1, 1, 1, 2, 1, 5, 7, 2, 1, 1, 1, 4, 1, 5, 1, 3, 2, 1, 1, 1, 1, 1, 16, 58, 4, 1, 3, 1, 63, 1, 83, 3, 1, 5, 2, 3, 3, 3, 4, 6, 3, 1, 3, 1, 5, 1, 1, 4, 1, 4, 1, 3, 5, 1, 13, 1, 3, 9, 1, 1, 1, 3, 1, 1, 1, 3, 1, 3, 1, 1, 22, 4, 4, 8, 1, 10, 1, 12, 34, 8, 2, 14, 1, 18, 2, 1, 7, 12, 1, 16, 2, 2, 1, 2, 2, 2, 4, 1, 19, 3, 1, 7, 4, 3, 1, 2, 1, 1, 21, 3, 10, 1, 3, 13, 1, 8, 1, 10, 3, 3, 1, 43, 4, 2, 17, 1, 4, 1, 4, 1, 1, 7, 1, 6, 1, 2, 7, 2, 4, 1, 6, 1, 1, 4, 2, 1, 8, 1, 1, 1, 1, 1, 12, 1, 1, 1, 101, 1, 2, 163, 2, 2, 2, 1, 2, 3, 1, 2, 8, 5, 1, 1, 1, 2, 4, 1, 2, 82, 8, 2, 1, 1, 8, 2, 1, 290, 1, 28, 3, 1, 3, 1, 6, 4, 1, 1, 1, 1, 68, 1, 2, 20, 1, 3, 49, 2, 3, 1, 26, 1, 17, 4, 1, 16, 1, 1, 2, 11, 2, 18, 18, 4, 1, 11, 4, 1, 2, 3, 8, 1, 6, 1, 2, 1, 9, 2, 7, 5, 2, 17, 2, 25, 1